# Spike: DuckDB reading the v1 tables straight from Iceberg

Run jobs can't read their inputs through the control plane's `/data` (a one-day filter
on `nwm30_medium_range` scanned the month and took the control plane down). Can a single
pod with DuckDB read Iceberg directly, pair and compute the metrics, a month at a time,
in bounded memory?

Two ways to read, each tried:

- **A. DuckDB's `iceberg` extension** attached to Polaris (the REST catalog) with your
  token and Polaris's vended storage credentials.
- **B. PyIceberg plans, DuckDB reads.** PyIceberg asks Polaris for the table, prunes
  files by partition, and hands over the file list and the vended credentials; DuckDB
  reads the Parquet files.

Both pair forecasts and observations through the crosswalk and compute per-gauge and
per-lead-day metrics in SQL (the shapes teehr-core's engine computes), over a day, a
week and a month of forecasts. Reports wall time, peak memory of this kernel and peak
spill to disk.

Reads only, as you. Self-contained. Pick a JupyterHub profile with more memory than
`MEMORY_LIMIT` below.

In [1]:
%pip install -q "duckdb>=1.4.1" "pyiceberg>=0.10" psutil

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil
import threading
import time
from pathlib import Path

import duckdb
import pandas as pd
import psutil
from pyiceberg.catalog import load_catalog
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan

duckdb.__version__

'2.0.0'

## 1. Settings

In [3]:
# Polaris (the Iceberg REST catalog), in-cluster. The catalog is named after the realm.
POLARIS = os.environ.get(
    "POLARIS_URL", "http://polaris.teehr-hub.svc.cluster.local:8181"
).rstrip("/")
REALM = os.environ.get("POLARIS_DEFAULT_REALM", "teehr")
REGION = os.environ.get("AWS_REGION", "us-east-1")
S3_ENDPOINT = os.environ.get("REMOTE_CATALOG_S3_ENDPOINT")  # KinD only (RustFS)

FORECAST, OBSERVED, CROSSWALK = "nwm30_medium_range", "usgs_streamflow", "usgs_nwm30"
WINDOWS = {  # step: (first reference_time, end (exclusive))
    "day": ("2024-01-01", "2024-01-02"),
    "week": ("2024-01-01", "2024-01-08"),
    "month": ("2024-01-01", "2024-02-01"),
}
MAX_LEAD = pd.Timedelta("16D")  # observations reach this far past a window

MEMORY_LIMIT = "8GB"  # DuckDB's; it spills past this
THREADS = 4
SPILL = Path("/tmp/duckdb_spill")

In [4]:
def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it).
    Polaris takes it as is (it maps your Keycloak roles to catalog roles)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


def window(step):
    start, end = (pd.Timestamp(t, tz="UTC") for t in WINDOWS[step])
    return start, end, end + MAX_LEAD


def ts(t):
    return f"TIMESTAMPTZ '{t:%Y-%m-%d %H:%M:%S}+00'"

In [5]:
class Peak:
    """Peak RSS of this kernel and peak size of DuckDB's spill directory, sampled."""

    def __init__(self, every=0.5):
        self.every, self.rss, self.spill = every, 0, 0
        self._stop = threading.Event()

    def _spill_bytes(self):
        if not SPILL.exists():
            return 0
        return sum(f.stat().st_size for f in SPILL.rglob("*") if f.is_file())

    def _sample(self):
        me = psutil.Process()
        while not self._stop.is_set():
            self.rss = max(self.rss, me.memory_info().rss)
            self.spill = max(self.spill, self._spill_bytes())
            time.sleep(self.every)

    def __enter__(self):
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()
        self.began = time.monotonic()
        return self

    def __exit__(self, *exc):
        self.seconds = time.monotonic() - self.began
        self._stop.set()
        self._thread.join()


def connect():
    shutil.rmtree(SPILL, ignore_errors=True)
    SPILL.mkdir(parents=True)
    con = duckdb.connect()
    con.execute(f"""
        SET memory_limit = '{MEMORY_LIMIT}';
        SET threads = {THREADS};
        SET temp_directory = '{SPILL}';
        SET TimeZone = 'UTC';
        INSTALL httpfs; LOAD httpfs;
        INSTALL iceberg; LOAD iceberg;
    """)
    return con

## 2. Pairing and metrics (the same SQL for both reads)

`{fc}`, `{obs}` and `{xw}` are whatever each read path names its tables.

In [6]:
def metrics_sql(fc, obs, xw, step):
    start, end, obs_end = window(step)
    return f"""
    WITH f AS (
        SELECT series_key, reference_time, value_time, value FROM {fc}
        WHERE reference_time >= {ts(start)} AND reference_time < {ts(end)}
    ), o AS (
        SELECT series_key, value_time, value FROM {obs}
        WHERE value_time >= {ts(start)} AND value_time < {ts(obs_end)}
    ), pairs AS (
        SELECT x.left_key AS gauge, o.value AS obs, f.value AS sim,
               CAST(floor(epoch(f.value_time - f.reference_time) / 86400) AS INTEGER)
                   AS lead_day
        FROM f
        JOIN {xw} x ON f.series_key = x.right_key
        JOIN o ON o.series_key = x.left_key AND o.value_time = f.value_time
    )
    SELECT gauge, lead_day, count(*) AS n,
           sqrt(avg(power(sim - obs, 2))) AS rmse,
           sum(sim - obs) / nullif(sum(obs), 0) AS relative_bias,
           1 - sum(power(sim - obs, 2))
               / nullif(sum(power(obs, 2)) - power(sum(obs), 2) / count(*), 0) AS nse,
           corr(sim, obs) AS r,
           1 - sqrt(power(corr(sim, obs) - 1, 2)
                    + power(stddev_pop(sim) / nullif(stddev_pop(obs), 0) - 1, 2)
                    + power(avg(sim) / nullif(avg(obs), 0) - 1, 2)) AS kge
    FROM pairs
    GROUP BY GROUPING SETS ((gauge), (lead_day))
    """


def measure(con, path, step, sql):
    with Peak() as peak:
        out = con.execute(sql).df()
    by_gauge = out[out.gauge.notna()]
    return {
        "path": path, "step": step, "seconds": round(peak.seconds, 1),
        "pairs": int(by_gauge.n.sum()), "gauges": len(by_gauge),
        "peak_rss_gb": round(peak.rss / 2**30, 2),
        "peak_spill_gb": round(peak.spill / 2**30, 2),
    }, out


results = []

## 3A. DuckDB's `iceberg` extension, attached to Polaris

If the attach or the first read fails, the error is printed and 3B still runs.

In [7]:
def attach(con):
    con.execute(f"CREATE OR REPLACE SECRET polaris (TYPE iceberg, TOKEN '{access_token()}')")
    con.execute(f"""
        ATTACH '{REALM}' AS cat (
            TYPE iceberg,
            ENDPOINT '{POLARIS}/api/catalog',
            SECRET polaris,
            ACCESS_DELEGATION_MODE 'vended_credentials'
        )
    """)


A_WORKS = False
try:
    con = connect()
    attach(con)
    print(con.execute(f"SELECT count(*) AS crosswalk_rows FROM cat.public.{CROSSWALK}").df())
    A_WORKS = True
except Exception as exc:  # noqa: BLE001 - a spike: show whatever went wrong
    print(f"A doesn't work here: {type(exc).__name__}: {exc}")

   crosswalk_rows
0            8518


In [8]:
if A_WORKS:
    for step in WINDOWS:
        con = connect()
        attach(con)  # a fresh token per step
        sql = metrics_sql(f"cat.public.{FORECAST}", f"cat.public.{OBSERVED}",
                          f"cat.public.{CROSSWALK}", step)
        try:
            row, out_a = measure(con, "A", step, sql)
        except Exception as exc:  # noqa: BLE001
            print(f"A {step}: {type(exc).__name__}: {exc}")
            break
        results.append(row)
        print(row)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'path': 'A', 'step': 'day', 'seconds': 3.4, 'pairs': 5882408, 'gauges': 6643, 'peak_rss_gb': 1.42, 'peak_spill_gb': 0.0}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'path': 'A', 'step': 'week', 'seconds': 11.2, 'pairs': 40055476, 'gauges': 6657, 'peak_rss_gb': 1.49, 'peak_spill_gb': 0.0}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'path': 'A', 'step': 'month', 'seconds': 44.5, 'pairs': 169337075, 'gauges': 7042, 'peak_rss_gb': 1.8, 'peak_spill_gb': 0.0}


## 3B. PyIceberg plans, DuckDB reads the files

PyIceberg asks Polaris for each table (as you, with vended credentials) and plans the
files the filter can match: whole partitions are skipped, and so are files whose column
stats rule them out. DuckDB then reads those Parquet files with the vended credentials.
(The copied tables have no delete files; the cell checks.)

In [9]:
def catalog():
    props = {
        "type": "rest",
        "uri": f"{POLARIS}/api/catalog",
        "warehouse": REALM,
        "token": access_token(),
        "header.X-Polaris-Realm": REALM,
        "header.X-Iceberg-Access-Delegation": "vended-credentials",
        "s3.region": REGION,
    }
    if S3_ENDPOINT:
        props |= {"s3.endpoint": S3_ENDPOINT, "s3.path-style-access": "true"}
    return load_catalog("polaris", **props)


def planned(cat, name, row_filter=None):
    """The table's data files the filter can match, and its storage credentials."""
    table = cat.load_table(("public", name))
    scan = table.scan(row_filter=row_filter) if row_filter is not None else table.scan()
    tasks = list(scan.plan_files())
    assert not any(t.delete_files for t in tasks), f"{name} has delete files"
    return table, [t.file.file_path for t in tasks]


def s3_secret(con, name, table):
    """A DuckDB S3 secret from the credentials Polaris vended for ``table``."""
    p = table.io.properties
    options = [
        "TYPE s3",
        f"KEY_ID '{p['s3.access-key-id']}'",
        f"SECRET '{p['s3.secret-access-key']}'",
        f"REGION '{p.get('s3.region') or p.get('client.region') or REGION}'",
        f"SCOPE '{table.metadata.location}'",
    ]
    if p.get("s3.session-token"):
        options.append(f"SESSION_TOKEN '{p['s3.session-token']}'")
    if S3_ENDPOINT:
        endpoint = S3_ENDPOINT.split("://", 1)[-1]
        options += [f"ENDPOINT '{endpoint}'", "URL_STYLE 'path'",
                    f"USE_SSL {str(S3_ENDPOINT.startswith('https')).lower()}"]
    con.execute(f"CREATE OR REPLACE SECRET {name} ({', '.join(options)})")


def files_sql(paths):
    return "read_parquet([" + ", ".join(f"'{p}'" for p in paths) + "])"


for step in WINDOWS:
    start, end, obs_end = window(step)
    cat = catalog()
    began = time.monotonic()
    fc, fc_files = planned(cat, FORECAST, And(
        GreaterThanOrEqual("reference_time", start.isoformat()),
        LessThan("reference_time", end.isoformat())))
    obs, obs_files = planned(cat, OBSERVED, And(
        GreaterThanOrEqual("value_time", start.isoformat()),
        LessThan("value_time", obs_end.isoformat())))
    xw, xw_files = planned(cat, CROSSWALK)
    plan_seconds = time.monotonic() - began
    print(f"B {step}: planned {len(fc_files)} forecast, {len(obs_files)} observation, "
          f"{len(xw_files)} crosswalk files in {plan_seconds:.1f}s")

    con = connect()
    for i, table in enumerate((fc, obs, xw)):
        s3_secret(con, f"vended_{i}", table)
    sql = metrics_sql(files_sql(fc_files), files_sql(obs_files), files_sql(xw_files), step)
    try:
        row, out_b = measure(con, "B", step, sql)
    except Exception as exc:  # noqa: BLE001
        print(f"B {step}: {type(exc).__name__}: {exc}")
        break
    row["plan_seconds"] = round(plan_seconds, 1)
    results.append(row)
    print(row)

B day: planned 22 forecast, 23 observation, 1 crosswalk files in 0.6s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'path': 'B', 'step': 'day', 'seconds': 2.7, 'pairs': 5882408, 'gauges': 6643, 'peak_rss_gb': 1.47, 'peak_spill_gb': 0.0, 'plan_seconds': 0.6}
B week: planned 22 forecast, 23 observation, 1 crosswalk files in 0.4s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'path': 'B', 'step': 'week', 'seconds': 11.1, 'pairs': 40055476, 'gauges': 6657, 'peak_rss_gb': 1.52, 'peak_spill_gb': 0.0, 'plan_seconds': 0.4}
B month: planned 22 forecast, 49 observation, 1 crosswalk files in 0.6s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'path': 'B', 'step': 'month', 'seconds': 43.3, 'pairs': 169337075, 'gauges': 7042, 'peak_rss_gb': 1.81, 'peak_spill_gb': 0.0, 'plan_seconds': 0.6}


## 4. Results

In [10]:
pd.DataFrame(results)

,path,step,seconds,pairs,gauges,peak_rss_gb,peak_spill_gb,plan_seconds
0,A,day,3.4,5882408,6643,1.42,0.0,NaN
1,A,week,11.2,40055476,6657,1.49,0.0,NaN
2,A,month,44.5,169337075,7042,1.80,0.0,NaN
3,B,day,2.7,5882408,6643,1.47,0.0,0.6
4,B,week,11.1,40055476,6657,1.52,0.0,0.4
5,B,month,43.3,169337075,7042,1.81,0.0,0.6


In [11]:
# The last step's metrics: per gauge, then per lead day.
last = out_b if "out_b" in globals() else out_a
display(last[last.gauge.notna()].drop(columns="lead_day").head(10))
display(last[last.lead_day.notna()].drop(columns="gauge").sort_values("lead_day"))

,gauge,n,rmse,relative_bias,nse,r,kge
0,usgs-05396000,3745,0.471633,0.079288,-0.547089,-0.146477,-0.303120
1,usgs-05551000,29440,11.031889,-0.074292,0.729024,0.870106,0.744825
2,usgs-05548105,18920,2.431042,-0.141052,0.282051,0.605571,0.544740
3,usgs-05550300,22040,1.746799,-0.411445,0.078083,0.615643,0.400131
4,usgs-05549000,20920,0.430473,-0.610408,-1.995317,0.272233,-0.232741
5,usgs-05541710,18520,14.670685,0.078806,0.118900,0.557655,0.550556
6,usgs-05543500,22120,417.673178,-0.195093,-0.282690,0.405227,0.372713
7,usgs-05529500,26520,1.397351,1.895521,-9.731818,0.646090,-1.900937
8,usgs-05532000,24056,1.357160,0.430082,-0.516954,0.461197,0.290438
9,usgs-05537500,21600,2.883058,-0.038183,0.107569,0.421340,0.339491


,lead_day,n,rmse,relative_bias,nse,r,kge
7043,0,16326943,148.064816,-0.057766,0.898970,0.948219,0.901007
7047,1,17006941,181.662970,-0.090498,0.858033,0.926387,0.859334
7049,2,16981726,196.072318,-0.106304,0.845559,0.919732,0.837494
7051,3,16956970,209.227404,-0.122789,0.835363,0.914521,0.812641
7052,4,16929364,220.627439,-0.133559,0.828280,0.910984,0.796241
7044,5,16904447,231.376031,-0.143237,0.822270,0.907912,0.783600
7050,6,16885888,243.068029,-0.154739,0.814641,0.903561,0.773685
7045,7,16876866,254.720694,-0.164679,0.806607,0.898886,0.765725
7046,8,16877108,265.125603,-0.169512,0.799648,0.894865,0.760928
7042,9,16884320,273.600341,-0.171273,0.794360,0.892018,0.754852
